In [1]:
import pandas as pd
import numpy as np
from cca_classes import ConvenienceYieldCCAPricer, compute_barrier_kvm, compute_lcl_usd 

## Baseline Model Panel

In [10]:
study_sovereigns = [
    'Saudi Arabia', 'UAE (Abu Dhabi)' , 'Qatar', 'Colombia',
    'Mexico', 'Brazil', 'Egypt', 'Malaysia','Indonesia', 'Philippines', 'Turkey', 'Chile', 'China',
    'South Africa', 'South Korea', 'Thailand']

cca_panel_df = pd.read_csv('../data/processed/CCA_V2/CCA_panel.csv')
cca_panel_df['date'] = pd.to_datetime(cca_panel_df['date'])
cca_panel_df = cca_panel_df[cca_panel_df['country'].isin(study_sovereigns)].copy()

T = 5.0
vol_window = 52
freq = 'W'
GAMMA = 0.05  # dampening parameter — adjust freely

cca_panel_df.set_index(['date','country'], inplace=True)
cca_panel_df = (
    cca_panel_df
    .groupby('country')
    .resample(freq, level='date')
    .last()
)
cca_panel_df.reset_index(inplace=True)


## M1 Drift Adjustment: Merge Futures & Compute Convenience Yield

In [11]:
# Load futures
oil_futures = pd.read_csv('../data/processed/Oil/oil_futures.csv')
oil_futures['date'] = pd.to_datetime(oil_futures['date'], format='%d.%m.%Y')
oil_futures = oil_futures.sort_values('date')


oil_prices = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv').sort_values('date')
oil_prices['date'] = pd.to_datetime(oil_prices['date'], format='%m/%d/%y')
oil_prices = oil_prices.sort_values('date')


# Ensure your main panel is also sorted by date
cca_panel_df = cca_panel_df.reset_index()
cca_panel_df = cca_panel_df.sort_values('date')

# 2. Perform the Directional Merge
# 'direction="nearest"' finds the closest date, whether it is before or after.
cca_panel_df = pd.merge_asof(
    cca_panel_df, 
    oil_prices[['date', 'Brent']], 
    on='date', 
    direction='nearest'
)

cca_panel_df = pd.merge_asof(
    cca_panel_df, 
    oil_futures[['date', 'Brent_12m']], 
    on='date', 
    direction='nearest'
)


# Compute convenience yield per row (using each country's own risk-free rate)
T_fut = 12/12
cca_panel_df['log_basis'] = np.log(cca_panel_df['Brent_12m'] / cca_panel_df['Brent'])
cca_panel_df['convenience_yield'] = cca_panel_df['risk_free_rate'] - (1/T_fut) * cca_panel_df['log_basis']

# Restore panel structure
cca_panel_df.set_index(['date', 'country'], inplace=True)
cca_panel_df = (
    cca_panel_df
    .groupby('country')
    .resample(freq, level='date')
    .last()
)
cca_panel_df.reset_index(inplace=True)


## Run M1

In [12]:
results = pd.DataFrame()

pricer = ConvenienceYieldCCAPricer(gamma=GAMMA)

for country, group in cca_panel_df.groupby('country'):

    df = group.copy().sort_values('date').reset_index(drop=True)

    r_d = df['domestic_rate']
    r_f = df['risk_free_rate']
    M_bn = df['monetary_base_bn_local']
    dom_D_bn = df['domestic_debt_bn_local']
    ext_D_bn = df['external_debt_bn_usd']
    fx_rate = df['fx_rate']
    y = df['convenience_yield']

    df['LCL_usd'] = [
        compute_lcl_usd(m, bd, fx, rd, rf, T)
        for m, bd, fx, rd, rf in zip(
            M_bn, dom_D_bn, fx_rate, r_d, r_f
        )
    ]

    ann_factor = np.sqrt(52) if freq == 'W' else np.sqrt(12)
    log_ret = np.log(df['LCL_usd'] / df['LCL_usd'].shift(1))
    df['sigma_lcl'] = log_ret.rolling(window=vol_window).std() * ann_factor

    # Rolling sigma_y from convenience yield (same window and annualization)
    dy = df['convenience_yield'].diff()
    df['sigma_y'] = dy.rolling(window=vol_window).std() * ann_factor

    df['B_f'] = [
        compute_barrier_kvm(debt, rf, T)
        for debt, rf in zip(ext_D_bn, r_f)
    ]

    out = {'implied_V': [], 'implied_sigma_V': [], 'cca_converged': []}
    for i, row in df.iterrows():
        cca = pricer.solve_CCA_M1(row['LCL_usd'], row['sigma_lcl'], row['B_f'],
                           r_f.iloc[i], y.iloc[i], row['sigma_y'], T)
        out['implied_V'].append(cca['implied_V'])
        out['implied_sigma_V'].append(cca['implied_sigma_V'])
        out['cca_converged'].append(cca['converged'])

    for col, vals in out.items():
        df[col] = vals

    results = pd.concat([results, df])

START_DATE = '2015-01-01'
END_DATE = '2024-12-31'

results = results[
    (results['date'] >= START_DATE) & (results['date'] <= END_DATE)
].copy()

# ── Save ──


/Users/juanfranciscoperez/commodities_and_sovereigns/Chapter_02/cca_classes.py:150: RuntimeWarning: overflow encountered in scalar multiply
  eq2 = Veff * sigma_total * norm.cdf(d1) - LCL_usd * sigma_lcl
/Users/juanfranciscoperez/commodities_and_sovereigns/Chapter_02/cca_classes.py:155: RuntimeWarning: overflow encountered in exp
  V       = np.exp(log_unknowns[0])
/Users/juanfranciscoperez/commodities_and_sovereigns/Chapter_02/cca_classes.py:156: RuntimeWarning: overflow encountered in exp
  sigma_V = np.exp(log_unknowns[1])
/Users/juanfranciscoperez/commodities_and_sovereigns/Chapter_02/cca_classes.py:146: RuntimeWarning: invalid value encountered in scalar divide
  d1 = (np.log(Veff / B_f) + (r_f + 0.5 * sigma_total**2) * T) / (sigma_total * sqt)


In [13]:
results[['date', 'country', 'cds_spread', 'risk_free_rate',
         'implied_V', 'implied_sigma_V', 'cca_converged',
         'B_f', 'LCL_usd', 'sigma_lcl',
         'convenience_yield',]].to_csv(
    '../output/final/M1_results_5YCDS_weekly_05_dampening.csv', index=False)